# 02_exploration_wttj : Welcome To The Jungle

Welcome to the Jungle (recommandé) => Site français, peu protégé, riche en offres tech : await page.goto("https://www.welcometothejungle.com/fr/jobs?refinementList%5Boffice_country_codes%5D%5B%5D=FR&query=data+engineer")

In [7]:
from playwright.async_api import async_playwright

In [8]:
'''Ce script simule un vrai navigateur Chrome pour accéder au site Welcome to the Jungle, 
puis interroge directement le moteur de recherche interne du site (Algolia) pour récupérer
les offres d'emploi Data Engineer en France. L'astuce centrale est d'utiliser le navigateur 
comme intermédiaire de confiance — Algolia refuse les requêtes venant d'un script Python externe, 
mais accepte celles venant d'un navigateur sur le domaine de WTTJ. Le script récupère les offres
page par page et les accumule dans une liste.'''

# Fonction principale — nb_pages contrôle combien de pages on récupère
# (20 offres par page, donc nb_pages=3 → 60 offres maximum)
async def scraper_wttj(nb_pages=3):

    # Liste qui accumulera toutes les offres récupérées
    toutes_offres = []

    # Démarrage de Playwright — le gestionnaire de navigateur
    async with async_playwright() as p:

        # Lancement d'un navigateur Chromium
        # headless=True → pas de fenêtre visible (mode silencieux)
        # disable-blink-features=AutomationControlled → masque le fait
        # que Chrome est piloté par un script (contourne certaines détections)
        browser = await p.chromium.launch(
            headless=True,
            args=["--disable-blink-features=AutomationControlled"]
        )

        # Création d'un contexte de navigation — comme un profil de navigateur
        # user_agent → on se fait passer pour un vrai Chrome sur Windows
        # viewport  → résolution d'écran standard d'un vrai utilisateur
        # locale    → langue française pour recevoir le bon contenu
        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            viewport={"width": 1920, "height": 1080},
            locale="fr-FR",
        )

        # Injection d'un script JavaScript exécuté avant chaque page
        # navigator.webdriver est une propriété que les sites utilisent
        # pour détecter les navigateurs automatisés — on la masque
        # en lui faisant retourner "undefined" au lieu de "true"
        await context.add_init_script("""
            Object.defineProperty(navigator, 'webdriver', {
                get: () => undefined
            });
        """)

        # Ouverture d'un nouvel onglet dans le contexte créé
        page = await context.new_page()

        # Navigation vers la page de recherche WTTJ
        # wait_until="domcontentloaded" → on attend que le HTML de base
        # soit chargé, sans attendre tous les appels réseau
        # (networkidle bloquerait indéfiniment sur ce site)
        await page.goto(
            "https://www.welcometothejungle.com/fr/jobs?"
            "refinementList%5Boffice_country_codes%5D%5B%5D=FR"
            "&query=data+engineer",
            wait_until="domcontentloaded"
        )

        # Pause de 2 secondes pour laisser le JavaScript s'initialiser
        # et établir la session avec Algolia
        await page.wait_for_timeout(2000)

        # Boucle sur chaque page de résultats
        for numero_page in range(nb_pages):
            print(f"Page {numero_page + 1}/{nb_pages}...")

            # Exécution d'un appel fetch() directement dans le navigateur
            # C'est la clé du fonctionnement : on appelle l'API Algolia
            # DEPUIS le navigateur qui est sur le domaine WTTJ
            # → Algolia accepte la requête car elle vient du bon domaine
            # f""" ... """ permet d'injecter la variable Python numero_page
            # dans le code JavaScript avec {numero_page}
            resultats = await page.evaluate(f"""
                async () => {{
                    const response = await fetch(
                        "https://CSEKHVMS53-dsn.algolia.net/1/indexes/wttj_jobs_production_fr/query",
                        {{
                            method: "POST",
                            headers: {{
                                // Identifiant de l'application Algolia de WTTJ
                                "X-Algolia-Application-Id": "CSEKHVMS53",
                                // Clé publique extraite des requêtes du navigateur
                                "X-Algolia-API-Key": "4bd8f6215d0cc52b26430765769e65a0",
                                "Content-Type": "application/json"
                            }},
                            body: JSON.stringify({{
                                query: "data engineer",  // mots recherchés
                                hitsPerPage: 20,         // 20 offres par page
                                page: {numero_page}      // numéro de page (0, 1, 2...)
                            }})
                        }}
                    );
                    // Conversion de la réponse HTTP en objet JSON Python
                    return await response.json();
                }}
            """)

            # Extraction des offres (hits) et du nombre total depuis la réponse
            hits    = resultats.get("hits", [])     # liste des offres de cette page
            nb_hits = resultats.get("nbHits", 0)    # nombre total d'offres disponibles

            print(f"  → {len(hits)} hits, {nb_hits} total")

            # Ajout des offres de cette page à la liste globale
            toutes_offres.extend(hits)

            # Pause de 0.5 seconde entre chaque requête
            # pour ne pas surcharger le serveur et éviter la détection
            await page.wait_for_timeout(500)

        # Fermeture du navigateur — libération des ressources
        await browser.close()

    # Retourne la liste complète de toutes les offres récupérées
    return toutes_offres

In [9]:
if __name__ == "__main__":
    
    offres = await scraper_wttj(nb_pages=1)
    print("offres ok")

    print(f"\n✅ {len(offres)} offres récupérées")
    print("Nombre offres ok")

    if offres:
        print("\nClés disponibles :")
        print(list(offres[0].keys()))
    print("Clés offres ok")

Page 1/1...
  → 20 hits, 1588 total
offres ok

✅ 20 offres récupérées
Nombre offres ok

Clés disponibles :
['organization', 'contract_type', 'has_contract_duration', 'rank_group_1', 'benefits', 'contract_duration_maximum', 'published_at_date', 'experience_level_minimum', 'contract_duration_minimum', 'offices', 'has_experience_level_minimum', 'slug', 'new_profession', 'key_missions', 'profile_ranking', 'rank_group_3', 'rank_group_2', 'salary_maximum', 'salary_period', 'salary_yearly_minimum', 'remote', 'has_education_level', 'is_boosted', 'education_level', 'summary', 'wk_reference', 'salary_currency', 'reference', 'has_salary_yearly_minimum', 'profile', 'name', 'has_remote', 'source_stage', 'organization_score', 'published_at', 'salary_minimum', 'language', 'sectors', 'published_at_timestamp', 'has_benefits', '_geoloc', 'objectID', '_highlightResult']
Clés offres ok
